In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
esm_model = esm_model.to(device)
esm_model.eval()

train_sample = train_df.sample(2000, random_state=42)
test_sample = test_df.sample(500, random_state=42)

y_train_esm = train_sample["label"].values
y_test_esm = test_sample["label"].values

def get_embedding(seq):
    inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = esm_model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

X_train_esm = np.array([get_embedding(seq) for seq in train_sample["seq"]])
X_test_esm = np.array([get_embedding(seq) for seq in test_sample["seq"]])

esm_model_ridge = Ridge()
esm_model_ridge.fit(X_train_esm, y_train_esm)

esm_preds = esm_model_ridge.predict(X_test_esm)

esm_mae = mean_absolute_error(y_test_esm, esm_preds)
esm_r2 = r2_score(y_test_esm, esm_preds)

print("ESM MAE:", esm_mae)
print("ESM R2:", esm_r2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'train_df' is not defined